In [1]:
from openai import OpenAI, api_key

In [ ]:
model="gpt-4.1-mini"
api_key="API"


In [ ]:
client=OpenAI(
    api_key=api_key,
    max_retries=6
)

In [ ]:
import requests

headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json'
}

messages = [
    {"role": "user", "content": "j"}
]
data = {
    'model': model,
    'messages': messages,
    'max_tokens': 1,
}

response = requests.post('https://api.openai.com/v1/chat/completions', headers=headers, json=data)
response_headers = response.headers['x-ratelimit-remaining-requests']
print("Remaining requests: ", response_headers)

In [ ]:
client.with_options(max_retries=5).chat.completions.create(
    messages=[
        {
        "role":"user",
        "content": "im very happy. Will you help me get better?",
        }
    ],
model=model
)

In [ ]:

from tenacity import retry, stop_after_attempt, wait_random_exponential

@retry(
    reraise=True,
    stop=stop_after_attempt(6),
    wait=wait_random_exponential(min=1, max=60)
)
def create_with_backoff(**kwargs):
    return client.responses.create(**kwargs)

resp = create_with_backoff(
    model=model,
    input=[{"role":"user","content":"Tell me a joke."}]
)
print(resp.output_text)

In [ ]:
import random
import time
import openai

def retry_with_backoff(
    func,
    initial_delay=1,
    factor=2,
    jitter=True,
    max_retries=5
):
    def wrapper(*args, **kwargs):
        delay = initial_delay
        for i in range(max_retries):
            try:
                return func(*args, **kwargs)
            except openai.RateLimitError as e:
                sleep = delay * (1 + (random.random() if jitter else 0))
                print(f"Rate limited, retrying in {sleep:.1f}s...")
                time.sleep(sleep)
                delay *= factor
        raise Exception("Max retries exceeded.")
    return wrapper

@retry_with_backoff
def create_manual(**kwargs):
    return client.responses.create(**kwargs)

resp3 = create_manual(
    model=model,
    input=[{"role":"user","content":"Explain Kubernetes."}]
)
print(resp3.output_text)